# CarBids Studio — auction-sale modelling

Choose `MODE="tabular"` for the public wholesale dataset (CPU is sufficient), or `MODE="enhanced"` for a source-audited, text-rich auction export (enable GPU and Internet).

This notebook never relabels asking prices or unsuccessful bids as sales. Labels are provider/platform-reported sale amounts, **not independently verified settlements**. USD only; fees/taxes/shipping are excluded unless separately documented. The wholesale dataset has no rich descriptions and cannot train MiniLM. No make/model transformer, TF-IDF, ridge or Extra Trees.

Outputs remain under `/kaggle/working/carbids`. Each enhanced optimizer fit checkpoints every ten updates; the eight-hour session budget stops gracefully, allowing outputs to save. Attach one prior saved notebook output as Input to resume with unchanged settings. This does not schedule new GPU sessions automatically.

In [ ]:
MODE='tabular'  # change to 'enhanced' only with an audited pre-sale-text export attached
import os,sys,time,io,base64,zipfile,hashlib,json,shutil,subprocess,signal
from pathlib import Path
assert MODE in {'tabular','enhanced'}
START=time.monotonic();ROOT=Path('/kaggle/working/carbids');ROOT.mkdir(parents=True,exist_ok=True)
packages=['numpy>=1.26,<3','pandas>=2.1,<3','scikit-learn>=1.4,<2','lightgbm>=4.5,<5','streamlit>=1.40,<2','requests>=2.31,<3']
if MODE=='enhanced':packages+=['transformers>=5,<6','huggingface-hub>=1,<2','safetensors>=0.4']
subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
if MODE=='enhanced':
    import torch
    assert torch.cuda.is_available(),'Enable a GPU accelerator; full transformer training is not silently sent to CPU.'
    print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
SOURCE_ZIP='...'
SOURCE_HASH='6ba8a5a27e50a157338a3274f8dc60c25c9edd96a4298544c68872aa91bb5606'
source=base64.b64decode(SOURCE_ZIP)
assert hashlib.sha256(source).hexdigest()==SOURCE_HASH
previous=list(Path('/kaggle/input').rglob('carbids/run-config.json'))
if len(previous)>1:raise ValueError('Attach just one prior CarBids output.')
if previous and not (ROOT/'run-config.json').exists():shutil.copytree(previous[0].parent,ROOT,dirs_exist_ok=True)
config={'mode':MODE,'source_hash':SOURCE_HASH}
state=ROOT/'run-config.json'
if state.exists() and json.loads(state.read_text())!=config:raise ValueError('Previous session has different code or mode; use separate outputs.')
with zipfile.ZipFile(io.BytesIO(source)) as z:z.extractall(ROOT)
state.write_text(json.dumps(config));os.chdir(ROOT);sys.path.insert(0,str(ROOT))
DATA=ROOT/'data/cohort.jsonl';DATA.parent.mkdir(parents=True,exist_ok=True)
OUTPUT=ROOT/('models/tabular' if MODE=='tabular' else 'artifacts/enhanced')


## Prepare all eligible sales

Tabular mode downloads the public wholesale CSV archive or uses an attached `car_prices.csv`. Enhanced mode needs exactly one `auction_export.csv` or `auction_export.jsonl` using the repository’s audited schema. Only explicit “Sold for” evidence matching the sale amount is accepted. Undated/post-sale text, unsuccessful bids, invalid VINs and non-USD targets are refused. One sale per VIN, duplicate-text removal and whole-date chronological splits precede fitting.

In [ ]:
from carbids.data import prepare,download,load_cohort
if not DATA.exists():
    if MODE=='tabular':
        attached=list(Path('/kaggle/input').rglob('car_prices.csv'))
        source=attached[0] if attached else download(ROOT/'data/auction_sales.zip')
        prepare(source,DATA,'wholesale_2015')
    else:
        exports=list(Path('/kaggle/input').rglob('auction_export.csv'))+list(Path('/kaggle/input').rglob('auction_export.jsonl'))
        if len(exports)!=1:raise ValueError('Attach exactly one audited auction_export.csv or auction_export.jsonl. See README and data schema.')
        prepare(exports[0],DATA,'auction_export')
d=load_cohort(DATA,require_text=MODE=='enhanced')
print('Eligible distinct sales:',len(d));del d


## Fit, assess, then refit

The baseline compares raw/log targets and three Huber thresholds. Enhanced mode first fine-tunes MiniLM-L6 (three OOF folds, two epochs per fit), supplies `predicted_text_price`, and filters **training** OOF prices with Isolation Forest auto. Validation/test stay unfiltered. Validation dollar MAE selects settings; test metrics are measured before the all-row deployment refit. SHAP explains the fitted LightGBM function, not causal effects or MiniLM words.

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True)
if MODE=='tabular':
    cmd=[sys.executable,'-u','-m','carbids.baseline','--data',str(DATA),'--output',str(OUTPUT),'--threads','4','--trees','700']
else:
    cmd=[sys.executable,'-u','-m','carbids.train','--data',str(DATA),'--output',str(OUTPUT),'--device','cuda','--threads','4','--folds','3','--epochs','2','--batch-size','4','--accumulation','2']
log=OUTPUT/'training.log';returncode=None
if not (OUTPUT/'manifest.json').exists():
    remaining=max(0,8*3600-(time.monotonic()-START))
    if remaining>300:
        with log.open('a',buffering=1) as stream:
            process=subprocess.Popen(cmd,cwd=ROOT,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
            deadline=time.monotonic()+remaining
            while process.poll() is None and time.monotonic()<deadline:
                time.sleep(30);print('\n'.join(log.read_text(errors='replace').splitlines()[-3:]),flush=True)
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGINT)
                try:process.wait(timeout=120)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGTERM);process.wait(timeout=30)
                print('Session budget reached; preserve outputs and attach them to resume.')
            else:returncode=process.returncode
status='ready' if (OUTPUT/'manifest.json').exists() else ('failed' if returncode not in (None,0) else 'pending')
(ROOT/'training-status.json').write_text(json.dumps({'status':status,'mode':MODE,'returncode':returncode},indent=2))
print('STATUS:',status)
if status=='failed':print(log.read_text(errors='replace')[-12000:])


In [ ]:
evaluation=OUTPUT/'evaluation.json'
if evaluation.exists():print(evaluation.read_text())
if (OUTPUT/'manifest.json').exists():
    from carbids.inference import PricePipeline
    model=PricePipeline(OUTPUT)
    print('Inference:',model.predict('Well maintained auction vehicle with service records, clean interior and disclosed scratches.',{'year':2014,'manufacturer':'toyota','model':'camry','odometer':65000,'condition':'4.0','source':model.manifest['sources'][0]},explain=True)['price'])
    destination=Path('/kaggle/working/CarBids_Trained.zip')
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT/'app.py','app.py')
        for name in ['requirements.txt','requirements-training.txt','requirements-api.txt']:z.write(ROOT/name,name)
        for file in (ROOT/'carbids').glob('*.py'):z.write(file,str(file.relative_to(ROOT)))
        for file in OUTPUT.rglob('*'):
            if file.is_file() and 'optimizer-checkpoints' not in file.parts and 'description-oof' not in file.parts and file.name!='assessment-lightgbm.txt':z.write(file,str(file.relative_to(ROOT)))
    print('Download:',destination)
else:print('Not complete. Save outputs, attach this notebook output to a new session, and rerun unchanged.')
